# 06: LGD Model Training

This notebook trains six regression models for Loss Given Default (LGD) estimation. Each model is tuned via RandomizedSearchCV on a random sample of 100,000 observations (from the defaulted loans of the development sample, matured loans only) and evaluated on a held-out test set. The model with the best **cross-validated** RMSE is selected for portfolio inference; the test metrics are reported but play no role in the choice.

**Input:** `data/processed/checkpoint_df_LGD_training.pkl`, `data/processed/features_LGD_num_for_training.pkl`, `data/processed/features_LGD_cat_for_training.pkl`, `data/processed/checkpoint_df_master_with_PD.pkl`  
**Output:** `models/<best_model>_LGD.pkl`, `models/LGD_model_comparison.csv`, `models/pipeline_LGD.pkl`, `data/processed/checkpoint_df_master_with_PD_LGD.pkl`

## Libraries

In [ ]:
# Run from the repository root so that data/, models/ and src/ resolve the same way
# whether the notebook is opened from the root or from production_notebooks/.
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "production_notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import shap
import os

from sklearn.model_selection import train_test_split

from train_models_LGD import train_all_models_LGD
from pipeline_LGD     import FeaturePipeline_LGD
from readme_figures   import plot_lgd_model_comparison
from explain          import shap_values_for

## 1. Load Checkpoint

In [ ]:
with open("data/processed/checkpoint_df_LGD_training.pkl", "rb") as f:
    data = pickle.load(f)
df_LGD_clean = data["df_LGD_clean"]        # defaulted, matured loans (EDA view of the treatment)

with open("data/processed/features_LGD_num_for_training.pkl", "rb") as f:
    num_features_LGD = pickle.load(f)

with open("data/processed/features_LGD_cat_for_training.pkl", "rb") as f:
    cat_features_LGD = pickle.load(f)

features_LGD = num_features_LGD + cat_features_LGD

# Raw features for every loan (with PD already computed): the LGD pipeline is
# fitted on the raw training rows and applied unchanged to test and portfolio.
with open("data/processed/checkpoint_df_master_with_PD.pkl", "rb") as f:
    df_master_features = pickle.load(f)["df_master_features"]

print(f"LGD development sample: {len(df_LGD_clean):,} defaulted, matured loans")
print(f"Features: {features_LGD}")

## 2. Train / Test Split

Random split of the defaulted development sample (no stratification, the regression target is continuous).

In [ ]:
idx_train, idx_test = train_test_split(df_LGD_clean.index, test_size=0.2, random_state=42)

y_train_LGD = df_LGD_clean.loc[idx_train, "Target_Variable_Loss"]
y_test_LGD  = df_LGD_clean.loc[idx_test, "Target_Variable_Loss"]

print(f"Train: {len(idx_train):,} | Test: {len(idx_test):,}")
print(f"Mean LGD train: {y_train_LGD.mean():.4f} | test: {y_test_LGD.mean():.4f}")

## 3. Preprocessing

`FeaturePipeline_LGD` is fitted on the raw features of the training split only: outlier and missing-value treatment from `LGD_TREATMENTS`, one-hot encoding of the categorical features and a RobustScaler on the numeric ones. The same fitted object transforms the test split here and the whole portfolio in section 6, so training and scoring cannot drift apart.

In [ ]:
pipeline_LGD = FeaturePipeline_LGD().fit(df_master_features.loc[idx_train, features_LGD])

X_train_LGD = pipeline_LGD.transform(df_master_features.loc[idx_train, features_LGD])
X_test_LGD  = pipeline_LGD.transform(df_master_features.loc[idx_test, features_LGD])

print(f"X_train_LGD shape: {X_train_LGD.shape} | X_test_LGD shape: {X_test_LGD.shape}")

## 4. Model Training

Six regression models are trained:
- Ridge Regression (interpretable baseline with L2 regularization)
- Decision Tree
- Random Forest
- XGBoost
- LightGBM
- CatBoost

All models tuned via RandomizedSearchCV (100k sample, 5-fold CV, optimizing RMSE). Results saved to `models/`.

In [ ]:
results_LGD = train_all_models_LGD(
    X_train_LGD, y_train_LGD,
    X_test_LGD,  y_test_LGD,
    sample_size=100_000
)

In [ ]:
# Results summary, ranked by cross-validated RMSE (the selection criterion).
# Test metrics are shown for reference only; they play no role in the choice.
results_LGD

In [ ]:
# Model comparison chart for the README (selected model highlighted)
plot_lgd_model_comparison(results_LGD, selected=results_LGD["cv_rmse"].idxmin())
plt.show()

## 5. SHAP Analysis

In [ ]:
# Load the model with the best cross-validated RMSE
best_name_LGD = results_LGD["cv_rmse"].idxmin()
with open(f"models/{best_name_LGD}_LGD.pkl", "rb") as f:
    model_LGD = pickle.load(f)
print(f"Selected model: {best_name_LGD} (CV RMSE {results_LGD.loc[best_name_LGD, 'cv_rmse']})")

# SHAP on a representative sample
# (LinearExplainer if a linear model is selected, TreeExplainer for a tree ensemble)
X_shap_LGD = pd.concat([X_train_LGD, X_test_LGD], axis=0).sample(n=20_000, random_state=42).astype(float)
shap_values_LGD = shap_values_for(model_LGD, X_shap_LGD, X_train_LGD)

print(f"SHAP values shape: {shap_values_LGD.shape}")

shap.summary_plot(shap_values_LGD, X_shap_LGD, max_display=X_shap_LGD.shape[1], show=False)
plt.savefig("assets/shap_lgd_final.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. LGD Inference on Full Portfolio

The LGD model is applied to the full portfolio, including non-defaulted and not yet matured loans, through the pipeline fitted on the training split. This is standard practice in Basel IRB: the question is what would be lost if each loan defaulted. Its main limitation is that performing loans may sit outside the feature ranges seen on defaulted loans, which the fitted caps only partly control. Predictions are clipped to [0, 100], the range of the target.

In [ ]:
# Whole portfolio through the pipeline fitted on the training split
X_full_LGD = pipeline_LGD.transform(df_master_features[features_LGD])
print(f"X_full_LGD shape: {X_full_LGD.shape}")

In [ ]:
# Inference, clipped to the range of the target
y_pred_raw = model_LGD.predict(X_full_LGD)
y_pred_LGD = np.clip(y_pred_raw, 0, 100)
df_master_features["LGD"] = y_pred_LGD

print(f"LGD computed for {len(y_pred_LGD):,} loans")
print(f"Predictions outside [0, 100] before clipping: {np.mean((y_pred_raw < 0) | (y_pred_raw > 100)):.4%}")
print(f"Mean LGD   : {y_pred_LGD.mean():.4f}")
print(f"Median LGD : {np.median(y_pred_LGD):.4f}")

In [ ]:
# Calibration check: predicted vs observed LGD where the outcome is complete
for name, idx in [("development sample (defaulted, matured)", df_LGD_clean.index),
                  ("test split", idx_test)]:
    pred = df_master_features.loc[idx, "LGD"]
    real = df_master_features.loc[idx, "Target_Variable_Loss"]
    print(f"{name}: mean predicted {pred.mean():.2f} | mean observed {real.mean():.2f} | "
          f"median predicted {pred.median():.2f} | median observed {real.median():.2f}")

In [ ]:
# Save pipeline and df_master_features with PD and LGD columns
pipeline_LGD.save("models/pipeline_LGD.pkl")

with open("data/processed/checkpoint_df_master_with_PD_LGD.pkl", "wb") as f:
    pickle.dump({"df_master_features": df_master_features}, f)

print("Saved pipeline and df_master_features with PD and LGD columns.")